# Aula 08 — Losses, logits e reduções

Vamos conferir MSE, BCE em logits e cross-entropy contra fórmulas independentes; depois investigar shapes, estabilidade, pesos, máscaras e denominadores.

Requer Python >=3.10, PyTorch >=2.6 e NumPy >=1.24. Validado em 9 de setembro de 2026 com Python 3.12.14, PyTorch 2.6.0+cpu e NumPy 2.3.5, CPU e float64. nbformat >=5.10 é usado somente para validar o arquivo. Execute todas as células em ordem, com estado reiniciado. Não há downloads, arquivos externos ou credenciais.

Dados sintéticos: valores explícitos e gerador NumPy com seed 20260908. Não há treinamento ou escolha de hiperparâmetros; portanto, os experimentos algébricos não estimam generalização. As contraprovas de instabilidade produzem não finitos deliberadamente, que são detectados por asserts; não há erro nem aviso inesperado.

In [ ]:
import math
import sys
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

SEED = 20260908
torch.manual_seed(SEED)
torch.set_num_threads(1)
rng = np.random.default_rng(SEED)
dtype = torch.float64
checks = []
def check(name, condition):
    assert bool(condition), name
    checks.append(name)
def rejects(name, function):
    try:
        function()
    except ValueError:
        check(name, True)
    else:
        raise AssertionError(name)
print('Python', sys.version.split()[0], '| PyTorch', torch.__version__, '| NumPy', np.__version__)
check('CPU', torch.empty(0).device.type == 'cpu')

## 1. MSE: média de elementos e o fator 1/2

Há quatro resíduos e soma dos quadrados 8. MSELoss devolve 2; o objetivo de meia MSE usado na Aula 07 devolve 1. O fator também multiplica o gradiente.

In [ ]:
prediction = torch.tensor([[1.,2.],[3.,4.]], dtype=dtype, requires_grad=True)
target = torch.tensor([[1.,0.],[1.,4.]], dtype=dtype)
unreduced = nn.MSELoss(reduction='none')(prediction, target)
mse = nn.MSELoss()(prediction, target)
g_mse, = torch.autograd.grad(mse, prediction)
g_half, = torch.autograd.grad(nn.MSELoss()(prediction,target)/2, prediction)
check('MSE none shape', unreduced.shape == target.shape)
check('MSE soma oito', unreduced.sum().item() == 8)
check('MSE mean dois', mse.item() == 2)
check('MSE fórmula de gradiente', torch.equal(g_mse, 2*(prediction.detach()-target)/target.numel()))
check('meia MSE divide gradiente', torch.equal(g_half, g_mse/2))
check('MSE sum coincide', nn.MSELoss(reduction='sum')(prediction,target).item() == 8)
print('MSE:', mse.item(), '| meia MSE:', mse.item()/2, '| gradiente:', g_mse.tolist())

## 2. Barrar broadcasting acidental antes da API

A subtração entre (3,1) e (3,) produz (3,3). Calculamos deliberadamente esse erro pela aritmética elementar, sem acionar o warning da API MSE. O contrato estrito rejeita o alvo antes do cálculo.

In [ ]:
def same_float_shape(a, b):
    if a.shape != b.shape or not a.is_floating_point() or not b.is_floating_point():
        raise ValueError('Predição e alvo devem ter shapes iguais e dtype flutuante')
    if a.device != b.device or not torch.isfinite(a).all() or not torch.isfinite(b).all():
        raise ValueError('Dispositivo incompatível ou valores não finitos')

column = torch.tensor([[1.],[2.],[3.]], dtype=dtype)
vector = torch.tensor([1.,2.,3.], dtype=dtype)
broadcast_wrong = (column-vector).square().mean().item()
rejects('shape incorreto rejeitado', lambda: same_float_shape(column,vector))
same_float_shape(column,vector[:,None])
check('broadcast criou pares indevidos', (column-vector).shape == (3,3))
check('MSE corrigida zero', F.mse_loss(column,vector[:,None]).item() == 0)
check('MSE incorreta quatro terços', abs(broadcast_wrong-4/3)<1e-14)
print('Erro por broadcasting:', broadcast_wrong, '| após corrigir shape: 0')

## 3. BCE em logits: paridade com NumPy

O alvo flutuante tem o mesmo shape do logit e fica entre zero e um. Sem pesos, a derivada da média é (sigmoid(z)-y)/N. A referência NumPy usa logaddexp para a softplus.

In [ ]:
z_np = np.array([[-2.,0.,2.],[1.,-1.,.5]])
y_np = np.array([[0.,1.,1.],[1.,0.,0.]])
z_bce = torch.tensor(z_np,dtype=dtype,requires_grad=True)
y_bce = torch.tensor(y_np,dtype=dtype)
same_float_shape(z_bce,y_bce)
bce_np = np.logaddexp(0,z_np)-y_np*z_np
bce_none = nn.BCEWithLogitsLoss(reduction='none')(z_bce,y_bce)
bce = nn.BCEWithLogitsLoss()(z_bce,y_bce)
g_bce, = torch.autograd.grad(bce,z_bce)
bce_error = float(np.max(np.abs(bce_none.detach().numpy()-bce_np)))
bce_gradient_error = float(np.max(np.abs(g_bce.numpy()-((1/(1+np.exp(-z_np))-y_np)/z_np.size))))
check('BCE none shape', bce_none.shape == z_bce.shape)
check('BCE NumPy', bce_error<1e-14)
check('BCE gradiente NumPy', bce_gradient_error<1e-14)
check('BCE média de seis elementos', torch.allclose(bce,bce_none.sum()/6,atol=1e-14,rtol=0))
check('BCE módulo versus função', torch.equal(bce,F.binary_cross_entropy_with_logits(z_bce,y_bce)))
print('BCE:', bce.item(), '| erro forward:', bce_error, '| erro gradiente:', bce_gradient_error)

## 4. Logits extremos: erro confiante deve continuar penalizado

Para z=-1000 com y=1 e z=1000 com y=0, cada loss estável é 1000. Calcular log de probabilidades arredondadas a zero/um perde informação. A contraprova não é uma recomendação de implementação.

In [ ]:
extreme_z = torch.tensor([-1000.,1000.],dtype=dtype,requires_grad=True)
extreme_y = torch.tensor([1.,0.],dtype=dtype)
stable_bce = F.binary_cross_entropy_with_logits(extreme_z,extreme_y,reduction='none')
stable_grad, = torch.autograd.grad(stable_bce.mean(),extreme_z)
p_extreme = torch.sigmoid(extreme_z.detach())
naive_bce = -(extreme_y*torch.log(p_extreme)+(1-extreme_y)*torch.log1p(-p_extreme))
check('BCE extremo finito', torch.isfinite(stable_bce).all())
check('BCE extremo mil', torch.equal(stable_bce,torch.full((2,),1000.,dtype=dtype)))
check('gradiente extremo preservado', torch.equal(stable_grad,torch.tensor([-.5,.5],dtype=dtype)))
check('contraprova BCE não finita detectada', not torch.isfinite(naive_bce).all())
print('BCE estável:', stable_bce.detach().tolist(), '| gradiente:',stable_grad.tolist(), '| ingênua:',naive_bce.tolist())

## 5. Cross-entropy: logits e índices de classe

Para B exemplos e C classes mutuamente exclusivas, logits têm shape (B,C), índices long têm shape (B). A soma sobre classes já faz parte da loss por exemplo; mean divide por B, sem pesos ou posições ignoradas.

In [ ]:
logits_np = np.array([[2.,1.,0.],[-1.,0.,1.],[.5,1.,-.5]])
labels_np = np.array([0,2,1])
shifted = logits_np-logits_np.max(axis=1,keepdims=True)
log_probs_np = shifted-np.log(np.exp(shifted).sum(axis=1,keepdims=True))
probs_np = np.exp(log_probs_np)
ce_np = -log_probs_np[np.arange(3),labels_np]
grad_ce_np = (probs_np-np.eye(3)[labels_np])/3
logits = torch.tensor(logits_np,dtype=dtype,requires_grad=True)
labels = torch.tensor(labels_np,dtype=torch.long)
ce_none = nn.CrossEntropyLoss(reduction='none')(logits,labels)
ce = nn.CrossEntropyLoss()(logits,labels)
g_ce, = torch.autograd.grad(ce,logits)
ce_error = float(np.max(np.abs(ce_none.detach().numpy()-ce_np)))
ce_gradient_error = float(np.max(np.abs(g_ce.numpy()-grad_ce_np)))
check('CE none remove eixo classes', ce_none.shape == (3,))
check('CE NumPy', ce_error<1e-14)
check('CE gradiente NumPy', ce_gradient_error<1e-14)
check('CE mean divide por três', torch.allclose(ce,ce_none.sum()/3,atol=1e-14,rtol=0))
check('CE equivale LogSoftmax NLL', torch.allclose(ce,nn.NLLLoss()(F.log_softmax(logits,dim=1),labels),atol=1e-14,rtol=0))
check('gradientes somam zero por exemplo', torch.allclose(g_ce.sum(dim=1),torch.zeros(3,dtype=dtype),atol=1e-14,rtol=0))
print('CE por exemplo:', ce_none.detach().tolist(), '| média:',ce.item(), '| erro gradiente:',ce_gradient_error)

## 6. Estabilidade e invariância da CE

Adicionar a mesma constante às classes de um exemplo não muda a distribuição, em aritmética exata. A verificação usa deslocamento 1000 em float64 e tolerância declarada. Isso não autoriza valores infinitos de entrada.

In [ ]:
shift_loss = F.cross_entropy(logits.detach()+1000,labels)
check('CE invariante a deslocamento moderado', torch.allclose(shift_loss,ce.detach(),atol=1e-12,rtol=0))
extreme_ce_z = torch.tensor([[1000.,0.,-1000.],[-1000.,0.,1000.]],dtype=dtype,requires_grad=True)
extreme_ce_y = torch.tensor([2,0],dtype=torch.long)
extreme_ce = F.cross_entropy(extreme_ce_z,extreme_ce_y,reduction='none')
extreme_ce_grad, = torch.autograd.grad(extreme_ce.mean(),extreme_ce_z)
naive_ce = -torch.log(F.softmax(extreme_ce_z.detach(),dim=1)[torch.arange(2),extreme_ce_y])
check('CE extremo dois mil', torch.equal(extreme_ce,torch.full((2,),2000.,dtype=dtype)))
check('CE extremo gradiente finito', torch.isfinite(extreme_ce_grad).all())
check('contraprova log softmax não finita detectada', not torch.isfinite(naive_ce).all())
print('CE estável extrema:',extreme_ce.detach().tolist(),'| ingênua:',naive_ce.tolist())

## 7. Dupla ativação muda a função objetivo

CrossEntropyLoss interpreta sua entrada como logits; BCEWithLogitsLoss também. Passar probabilidades a essas funções reaplica a transformação. Os resultados podem continuar finitos, tornando o defeito menos óbvio.

In [ ]:
double_ce = F.cross_entropy(F.softmax(logits,dim=1),labels)
double_bce = F.binary_cross_entropy_with_logits(torch.sigmoid(z_bce),y_bce)
check('softmax extra muda CE', abs(double_ce.item()-ce.item())>.01)
check('sigmoid extra muda BCE', abs(double_bce.item()-bce.item())>.01)
print('CE correta / dupla softmax:',ce.item(),double_ce.item())
print('BCE correta / dupla sigmoid:',bce.item(),double_bce.item())

## 8. Binário com um logit e com dois logits

BCE(z,y) equivale à CE([0,z],y) para y em {0,1}, sem pesos ou smoothing e com a mesma redução. Dois logits livres têm uma redundância de deslocamento; a equivalência de losses não garante trajetórias idênticas de otimizadores parametrizados diferentemente.

In [ ]:
binary_z = torch.tensor([-2.,0.,2.],dtype=dtype,requires_grad=True)
binary_y = torch.tensor([0,1,1],dtype=torch.long)
one_logit = F.binary_cross_entropy_with_logits(binary_z,binary_y.to(dtype))
two_logits = F.cross_entropy(torch.stack([torch.zeros_like(binary_z),binary_z],dim=1),binary_y)
g_one, = torch.autograd.grad(one_logit,binary_z)
g_two, = torch.autograd.grad(two_logits,binary_z)
check('BCE CE binária loss equivalente', torch.allclose(one_logit,two_logits,atol=1e-14,rtol=0))
check('BCE CE binária gradiente equivalente', torch.allclose(g_one,g_two,atol=1e-14,rtol=0))
print('BCE de um logit = CE de [0,z]:',one_logit.item())

## 9. Alvos: validar sem confiar apenas na execução

Os validadores abaixo se restringem ao caso tabular 2D. Alvos probabilísticos de CE precisam ser distribuições por linha. Uma loss finita não prova que os alvos têm semântica válida.

In [ ]:
def validate_indices(z,y):
    if z.ndim!=2 or y.shape!=(z.shape[0],) or y.dtype!=torch.long:
        raise ValueError('Esperados logits (B,C) e índices long (B,)')
    if not torch.isfinite(z).all() or not ((y>=0)&(y<z.shape[1])).all():
        raise ValueError('Logits não finitos ou índice fora das classes')

def validate_probabilities(z,q):
    same_float_shape(z,q)
    if z.ndim!=2 or not ((q>=0)&(q<=1)).all() or not torch.allclose(q.sum(1),torch.ones(z.shape[0],dtype=q.dtype,device=q.device),atol=1e-12,rtol=0):
        raise ValueError('Alvos devem ser distribuições por linha')

validate_indices(logits,labels)
rejects('CE rejeita coluna de índices',lambda:validate_indices(logits,labels[:,None]))
rejects('CE rejeita índice float',lambda:validate_indices(logits,labels.to(dtype)))
rejects('CE rejeita classe inexistente',lambda:validate_indices(logits,torch.tensor([0,2,3])))
onehot = F.one_hot(labels,num_classes=3).to(dtype)
validate_probabilities(logits,onehot)
rejects('CE rejeita distribuição sem soma um',lambda:validate_probabilities(logits,onehot*2))
check('onehot sem pesos equivale a índices',torch.allclose(F.cross_entropy(logits,onehot),ce,atol=1e-14,rtol=0))

## 10. CE ponderada e ignore_index: denominador explícito

Sem smoothing e com índices, mean divide pela soma dos pesos das classes observadas, excluindo posições ignoradas. A posição ignorada tem loss e gradiente zero. Uma máscara vazia deve ser tratada antes de dividir.

In [ ]:
weights = torch.tensor([1.,2.,3.],dtype=dtype)
masked_labels = torch.tensor([0,2,-100],dtype=torch.long)
valid = masked_labels!=-100
weighted_none = F.cross_entropy(logits,masked_labels,weight=weights,ignore_index=-100,reduction='none')
denominator = weights[masked_labels[valid]].sum()
weighted_ce = F.cross_entropy(logits,masked_labels,weight=weights,ignore_index=-100)
weighted_grad, = torch.autograd.grad(weighted_ce,logits)
check('denominador CE ponderada quatro',denominator.item()==4)
check('CE ponderada usa soma dos pesos',torch.allclose(weighted_ce,weighted_none.sum()/denominator,atol=1e-14,rtol=0))
check('posição ignorada loss zero',weighted_none[-1].item()==0)
check('posição ignorada gradiente zero',torch.count_nonzero(weighted_grad[-1]).item()==0)
check('média ingênua não equivale à ponderada',not torch.allclose(weighted_none.mean(),weighted_ce))
def masked_mean(values,mask):
    if values.shape!=mask.shape or mask.dtype!=torch.bool or not mask.any():
        raise ValueError('Máscara incompatível ou sem posições válidas')
    return values[mask].mean()
rejects('máscara vazia rejeitada',lambda:masked_mean(weighted_none,torch.zeros(3,dtype=torch.bool)))
print('CE ponderada:',weighted_ce.item(),'| denominador:',denominator.item(),'| média ingênua:',weighted_none.mean().item())

## 11. Alvos densos e label smoothing

Com probabilidades densas e pesos, CE mean divide pelo número de exemplos; com índices e pesos, divide pela soma dos pesos selecionados. Mesmo one-hot pode, portanto, mudar a escala. Smoothing é demonstrado sem pesos: q=(1-epsilon)*onehot+epsilon/C.

In [ ]:
dense_weighted = F.cross_entropy(logits,onehot,weight=weights)
index_weighted = F.cross_entropy(logits,labels,weight=weights)
dense_manual = -(onehot*weights*F.log_softmax(logits,dim=1)).sum(dim=1).mean()
dense_ratio = dense_weighted.item()/index_weighted.item()
check('CE densa ponderada fórmula',torch.allclose(dense_weighted,dense_manual,atol=1e-14,rtol=0))
check('onehot ponderado muda denominador',abs(dense_ratio-2)<1e-14)
epsilon=.1
q = (1-epsilon)*onehot+epsilon/3
validate_probabilities(logits,q)
smoothed = F.cross_entropy(logits,labels,label_smoothing=epsilon)
check('smoothing equivale a alvo misturado sem pesos',torch.allclose(smoothed,F.cross_entropy(logits,q),atol=1e-14,rtol=0))
g_smooth, = torch.autograd.grad(smoothed,logits)
check('gradiente smoothing',torch.allclose(g_smooth,(logits.detach().softmax(1)-q)/3,atol=1e-14,rtol=0))
print('CE ponderada densa / índices:',dense_weighted.item(),index_weighted.item(),'| razão:',dense_ratio)
print('CE com smoothing:',smoothed.item())

## 12. BCE: weight e pos_weight têm papéis distintos

Na fixture multilabel (2,3), pos_weight de shape (3,) pondera somente o termo positivo por classe. weight de shape (2,1) pondera cada exemplo e é difundido sobre suas três classes. mean divide pelos seis elementos, não pela soma de pesos.

In [ ]:
positive_weights = torch.tensor([1.,2.,3.],dtype=dtype)
sample_weights = torch.tensor([[1.],[4.]],dtype=dtype)
bce_weighted_none = F.binary_cross_entropy_with_logits(z_bce,y_bce,pos_weight=positive_weights,weight=sample_weights,reduction='none')
manual_weighted_bce = sample_weights*(positive_weights*y_bce*F.softplus(-z_bce)+(1-y_bce)*F.softplus(z_bce))
bce_weighted = F.binary_cross_entropy_with_logits(z_bce,y_bce,pos_weight=positive_weights,weight=sample_weights)
check('BCE pesos fórmula estável',torch.allclose(bce_weighted_none,manual_weighted_bce,atol=1e-14,rtol=0))
check('BCE ponderada divide por elementos',torch.allclose(bce_weighted,bce_weighted_none.sum()/6,atol=1e-14,rtol=0))
check('BCE não divide pela soma dos pesos expandidos',not torch.allclose(bce_weighted,bce_weighted_none.sum()/sample_weights.expand_as(z_bce).sum()))
g_weighted_bce, = torch.autograd.grad(bce_weighted,z_bce)
expected_g = sample_weights*((1-y_bce)*torch.sigmoid(z_bce.detach())-positive_weights*y_bce*(1-torch.sigmoid(z_bce.detach())))/6
check('BCE ponderada gradiente',torch.allclose(g_weighted_bce,expected_g,atol=1e-14,rtol=0))
print('BCE ponderada:',bce_weighted.item(),'| divisor da API:',z_bce.numel())

## 13. Eixo de classes e posições mascaradas

Uma saída (B,T,C) de Linear tem classes no último eixo. CE no caso multidimensional espera (B,C,T). O tensor abaixo é sintético; T representa posições e não requer estudar uma arquitetura sequencial agora.

In [ ]:
position_logits = torch.tensor(rng.normal(size=(2,4,3)),dtype=dtype,requires_grad=True)
position_targets = torch.tensor([[0,1,2,-100],[2,0,-100,-100]],dtype=torch.long)
position_valid = position_targets!=-100
position_none = F.cross_entropy(position_logits.movedim(-1,1),position_targets,ignore_index=-100,reduction='none')
position_mean = F.cross_entropy(position_logits.movedim(-1,1),position_targets,ignore_index=-100)
position_flat = F.cross_entropy(position_logits.reshape(-1,3),position_targets.reshape(-1),ignore_index=-100)
check('CE por posição tem shape B T',position_none.shape==(2,4))
check('cinco posições válidas',position_valid.sum().item()==5)
check('CE multidimensional equivale flatten correto',torch.allclose(position_mean,position_flat,atol=1e-14,rtol=0))
check('máscara normaliza por válidos',torch.allclose(position_mean,masked_mean(position_none,position_valid),atol=1e-14,rtol=0))
masked_dilution = position_none.mean().item()/position_mean.item()
check('média incluindo padding dilui por 5/8',abs(masked_dilution-5/8)<1e-14)
print('CE por posição:',position_mean.item(),'| média com padding:',position_none.mean().item(),'| razão:',masked_dilution)

## 14. Agregar lotes com o denominador correto

As três observações da fixture CE são separadas em lotes de tamanhos dois e um, sem atualizar parâmetros. Para CE ponderada com índices, agregue numeradores e somas dos pesos; ponderar apenas por quantidade de exemplos não resolve em geral.

In [ ]:
parts = [slice(0,2),slice(2,3)]
part_losses = [F.cross_entropy(logits[s],labels[s],weight=weights) for s in parts]
part_denoms = [weights[labels[s]].sum() for s in parts]
aggregate = sum(l*d for l,d in zip(part_losses,part_denoms))/sum(part_denoms)
naive_aggregate = torch.stack(part_losses).mean()
aggregation_error = abs(aggregate.item()-index_weighted.item())
naive_aggregation_error = abs(naive_aggregate.item()-index_weighted.item())
check('agregação ponderada coincide',aggregation_error<1e-14)
check('média simples de lotes diverge',naive_aggregation_error>1e-3)
print('Erro da agregação correta:',aggregation_error,'| erro da média de lotes:',naive_aggregation_error)

## 15. Auditoria final

Perdas finitas e derivadas corretas não substituem a escolha da tarefa e dos alvos. Os asserts conferem contratos e contraprovas, sem afirmar ganho de acurácia ou desempenho em produção.

In [ ]:
check('nomes de checks únicos',len(checks)==len(set(checks)))
check('objetivos principais finitos',all(math.isfinite(v.item()) for v in [mse,bce,ce,weighted_ce,smoothed,bce_weighted,position_mean]))
print(f'{len(checks)}/{len(checks)} verificações aprovadas')
print('Maiores erros BCE / CE:',bce_error,ce_error)
print('Maiores erros de gradiente BCE / CE:',bce_gradient_error,ce_gradient_error)
print('Contraprovas não finitas foram esperadas e detectadas; nenhuma usada em treinamento.')

## Exercícios e respostas

1. A MSE da Aula 07 contém 1/2. Basta usar MSELoss? Não: multiplique o resultado por 0,5 para manter loss e gradientes.
2. CE divide por B*C? Não no caso tabular simples: cada linha já produz uma loss, e mean divide por B.
3. BCE multilabel exige soma um por linha? Não. Cada posição representa uma decisão binária; o alvo precisa estar em [0,1].
4. É correto passar softmax à CE? Não: a entrada esperada é o logit. Use log_softmax com NLLLoss se quiser a decomposição estável.
5. Uma posição mascarada zero pode entrar no denominador? Só se esse for o objetivo deliberado; a média por posições válidas exclui-a também do divisor.
6. CE ponderada com one-hot e com índices sempre coincide? A loss unreduced coincide para one-hot, mas a média pode ter denominadores distintos.
7. pos_weight muda a definição de probabilidade calibrada? Muda o risco otimizado; decisões e calibração exigem avaliação própria, não uma promessa automática.
8. Que informação salvar para agregar métricas? Soma das perdas ponderadas e denominador correspondente, além da definição do objetivo.

## Fontes e continuação

Documentação oficial PyTorch 2.6, verificada em 9 de setembro de 2026:

- [MSELoss](https://docs.pytorch.org/docs/2.6/generated/torch.nn.MSELoss.html)
- [BCEWithLogitsLoss](https://docs.pytorch.org/docs/2.6/generated/torch.nn.BCEWithLogitsLoss.html)
- [CrossEntropyLoss](https://docs.pytorch.org/docs/2.6/generated/torch.nn.CrossEntropyLoss.html)
- [NLLLoss](https://docs.pytorch.org/docs/2.6/generated/torch.nn.NLLLoss.html)

Próxima: **Aula 09 — Dataset e transformações sem vazamento**. A escolha da unidade de análise e a procedência dos alvos passarão a fazer parte do pipeline de dados.